## Evaluation of Chunkers and embedding models

In [1]:
!pip install huggingface_hub[hf_xet]


[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from chunkers import (
    SentenceChunker,
    CharacterChunker,
    TokenChunker,
    RecursiveCharacterChunker,
    ResTokenChunker,
    KamradtChunker,
    ClusterChunker,
    LLMChunker
)
import os, re, sys
from chromadb.utils import embedding_functions
from chunking_evaluation import SyntheticEvaluation
import pandas as pd
from IPython.display import display, clear_output
from tabulate import tabulate
sys.path.append(os.path.abspath("../../../"))
from backend.database.core.funcs import get_documents_by_theme,get_document_themes
from backend.database.config.config import settings

In [3]:
def get_config_name(chunker,embed) -> str:
    print(chunker,embed)
    chunker_params = chunker.__getstate__()
    chunker_keys = '_'.join(str(chunker_params[key]) for key in chunker_params.keys())
    embed_name = embed.model_name if hasattr(embed,'model_name') else embed.__class__.__name__
    return f"{chunker.__class__.__name__}_{chunker_keys}_{embed_name}"

### List of chunkers for evaluation

In [4]:
chunkers = [
    # SentenceChunker configs
    SentenceChunker(sentences_per_chunk=1),
    SentenceChunker(sentences_per_chunk=2),
    SentenceChunker(sentences_per_chunk=5),
    SentenceChunker(sentences_per_chunk=10),
    SentenceChunker(sentences_per_chunk=15),
    SentenceChunker(sentences_per_chunk=20),

    # CharacterChunker configs
    CharacterChunker(characters_per_chunk=100, overlap=0),
    CharacterChunker(characters_per_chunk=200, overlap=50),
    CharacterChunker(characters_per_chunk=500, overlap=100),
    CharacterChunker(characters_per_chunk=1000, overlap=200),

    # TokenChunker configs
    TokenChunker(tokens_per_chunk=100, overlap=0),
    TokenChunker(tokens_per_chunk=200, overlap=20),
    TokenChunker(tokens_per_chunk=500, overlap=50),
    TokenChunker(tokens_per_chunk=1000, overlap=100),
    TokenChunker(tokens_per_chunk=800, overlap=400),
    TokenChunker(tokens_per_chunk=400, overlap=200),

    # RecursiveCharacterChunker configs
    RecursiveCharacterChunker(characters_per_chunk=100, overlap=0),
    RecursiveCharacterChunker(characters_per_chunk=200, overlap=50),
    RecursiveCharacterChunker(characters_per_chunk=500, overlap=100),
    RecursiveCharacterChunker(characters_per_chunk=1000, overlap=200),
    RecursiveCharacterChunker(characters_per_chunk=800, overlap=400),
    RecursiveCharacterChunker(characters_per_chunk=400, overlap=200),
    RecursiveCharacterChunker(characters_per_chunk=400, overlap=0),

    # # RecursiveTokenChunker configs
    ResTokenChunker(tokens_per_chunk=100, overlap=0),
    ResTokenChunker(tokens_per_chunk=200, overlap=20),
    ResTokenChunker(tokens_per_chunk=500, overlap=50),
    ResTokenChunker(tokens_per_chunk=1000, overlap=100),

    # # KamradtChunker configs
    KamradtChunker(avg_chunk_size=100, min_chunk_size=20),
    KamradtChunker(avg_chunk_size=200, min_chunk_size=50),
    KamradtChunker(avg_chunk_size=300, min_chunk_size=0),
    KamradtChunker(avg_chunk_size=500, min_chunk_size=100),
    KamradtChunker(avg_chunk_size=1000, min_chunk_size=200),

    # # ClusterChunker configs
    ClusterChunker(chunk_size=100),
    ClusterChunker(chunk_size=200),
    ClusterChunker(chunk_size=500),
    ClusterChunker(chunk_size=1000),

    # LLMChunker configs
    LLMChunker(model="gpt-3.5-turbo"),
    LLMChunker(model="gpt-4"),
    LLMChunker(model="gpt-4o"),
]

### List of embedding models for evaluation

In [5]:
embedding_models = [
    # embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2"),
    # embedding_functions.SentenceTransformerEmbeddingFunction(model_name="intfloat/multilingual-e5-large"),
    # embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-mpnet-base-v2"),
    # embedding_functions.SentenceTransformerEmbeddingFunction(model_name="BAAI/bge-m3"),
    # embedding_functions.SentenceTransformerEmbeddingFunction(model_name="bert-base-uncased"),
    # embedding_functions.SentenceTransformerEmbeddingFunction(model_name="nlpaueb/legal-bert-base-uncased"),
    # embedding_functions.SentenceTransformerEmbeddingFunction(model_name="distilbert-base-uncased"),
    # embedding_functions.OpenAIEmbeddingFunction(api_key=settings.OPENAI_API_KEY, model_name="text-embedding-3-large"),
    # embedding_functions.OpenAIEmbeddingFunction(api_key=settings.OPENAI_API_KEY, model_name="text-embedding-3-small"),
    embedding_functions.SentenceTransformerEmbeddingFunction(model_name="IoannisKat1/all-mpnet-base-v2-ft-new"),
    embedding_functions.SentenceTransformerEmbeddingFunction(model_name="IoannisKat1/bge-m3-ft-new"),
    embedding_functions.SentenceTransformerEmbeddingFunction(model_name="IoannisKat1/legal-bert-base-uncased-ft-new"),
    embedding_functions.SentenceTransformerEmbeddingFunction(model_name="IoannisKat1/all-MiniLM-L6-v2-ft-new"),
    embedding_functions.SentenceTransformerEmbeddingFunction(model_name="IoannisKat1/multilingual-e5-large-ft-new"),
    embedding_functions.SentenceTransformerEmbeddingFunction(model_name="IoannisKat1/distilbert-base-uncased-ft-new"),
    embedding_functions.SentenceTransformerEmbeddingFunction(model_name="IoannisKat1/bert-base-uncased-ft-new"),
    embedding_functions.SentenceTransformerEmbeddingFunction(model_name="IoannisKat1/modernbert-embed-base-ft-new"),
]

c:\Users\johnk\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


You are trying to use a model that was created with Sentence Transformers version 5.1.2, but you're currently using version 4.1.0. This might cause unexpected behavior or errors. In that case, try to update to the latest version.
You are trying to use a model that was created with Sentence Transformers version 5.1.2, but you're currently using version 4.1.0. This might cause unexpected behavior or errors. In that case, try to update to the latest version.
You are trying to use a model that was created with Sentence Transformers version 5.1.2, but you're currently using version 4.1.0. This might cause unexpected behavior or errors. In that case, try to update to the latest version.
You are trying to use a model that was created with Sentence Transformers version 5.1.2, but you're currently using version 4.1.0. This might cause unexpected behavior or errors. In that case, try to update to the latest version.
You are trying to use a model that was created with Sentence Transformers versio

### Evaluation per document theme

In [6]:
def eval_per_theme(theme:str,en_folders_path:str,path_folder:str,total_combinations:int):
    results = []
    if not os.path.exists(f'{en_folders_path}//synthetic_data_{theme}.csv'): return 
    current_combination = 0
    queries_csv_path = f'{en_folders_path}//synthetic_data_{theme}.csv'
    df = pd.read_csv(queries_csv_path)

    data = []
    for i in range(len(df)):
        location = df['corpus_id'][i].split('/')[-1]
        path_file = path_folder + f'/files/{theme}/{location}'
        data.append([
            df['question'][i],
            df['references'][i],
            path_file
        ])
    df = pd.DataFrame(data,columns=['question','references','corpus_id'])
    df.to_csv(f'{en_folders_path}//synthetic_data_{theme}.csv')

    corpora_paths = [f'{path_folder}/files/{theme}/{file}' for file in os.listdir(f'{path_folder}/files/{theme}')]
    evaluation = SyntheticEvaluation(
        corpora_paths=corpora_paths,
        queries_csv_path=queries_csv_path,
        openai_api_key=os.environ['OPENAI_API_KEY']
    )
    for chunker in chunkers:
        for embedding_model in embedding_models:
            current_combination += 1
            try:
                print(f"Evaluation combination {current_combination}/{total_combinations}:")
                print(f"Chunker: {chunker.__class__.__name__} (params: {str(chunker.__getstate__())})")
                print(f"Embedding: {embedding_model.model_name if hasattr(embedding_model,'model_name') else embedding_model.__class__.__name__}")

                result = evaluation.run(
                    chunker = chunker,
                    embedding_function = embedding_model,
                    retrieve = 5,
                    db_to_save_chunks=None
                )

                result['chunker'] = chunker.__class__.__name__
                result['embedding_function'] = embedding_model.model_name if hasattr(embedding_model,'model_name') else embedding_model.__class__.__name__
                result['config'] = get_config_name(chunker, embedding_model)
                results.append(result)
                clear_output(wait=True)

            except Exception as e:
                print(f"Error in combination {current_combination}:{str(e)}")
                continue

    df = pd.DataFrame(results)
    # df.to_csv('Chunking_Embedding_eval_all_queries_openai_embeddings.csv')
    df.to_csv(f'{path_folder}chunker_and_embedding_selection//Chunking_Embedding_eval_trained_embeddings_{theme}.csv')




In [7]:
total_combinations = len(chunkers) * len(embedding_models)

themes = get_document_themes()
results = []

p_folder = f'{os.getcwd()}'.split('\\')[:-1]
path_folder = '//'.join(p for p in p_folder)+'//'
en_folders_path = path_folder + 'synthetic_data_new'

eval_per_theme("Greek Cybercrime Legislation",en_folders_path,path_folder,total_combinations)

Evaluation combination 312/312:
Chunker: LLMChunker (params: {'model': 'gpt-4o'})
Embedding: IoannisKat1/modernbert-embed-base-ft-new


Processing chunks:  83%|████████▎ | 5/6 [00:00<00:00,  7.43it/s]


✅ Creating collection...
✅ Collection created.
✅ Preparing filtered questions...
✅ Valid questions count: 420 / 420
✅ Successfully added batch 0–50
✅ Successfully added batch 50–100
✅ Successfully added batch 100–150
✅ Successfully added batch 150–200
✅ Successfully added batch 200–250
✅ Successfully added batch 250–300
✅ Successfully added batch 300–350
✅ Successfully added batch 350–400
✅ Successfully added batch 400–420
<chunkers.LLMChunker object at 0x000002D98D060500> <chromadb.utils.embedding_functions.sentence_transformer_embedding_function.SentenceTransformerEmbeddingFunction object at 0x000002D9CAF05250>


### Result Showcase

In [10]:
def print_top(df_sorted:pd.DataFrame, metric_name:str):
    row = df_sorted.iloc[0]
    print(f"🏆 Top by {metric_name}:")
    print(f"{row['chunker']} | {row['embedding_function']} | {row['config']}")
    print(f"IOU Mean: {round(row['iou_mean']*100, 3)}%")
    print(f"IOU Std: {round(row['iou_std']*100, 3)}%")
    print(f"Recall Mean: {round(row['recall_mean']*100, 3)}%")
    print(f"Recall Std: {round(row['recall_std']*100, 3)}%")
    print(f"Precision Mean: {round(row['precision_mean']*100, 3)}%")
    print(f"Precision Std: {round(row['precision_std']*100, 3)}%")
    print(f"Precision Ω Mean: {round(row['precision_omega_mean']*100, 3)}%")
    print(f"Precision Ω Std: {round(row['precision_omega_std']*100, 3)}%")
    print("-" * 120)

def showcase_result(file_name:str):
    print(file_name)

    print("------------------------------------------------------------------------------------------------------------")

    df = pd.read_csv(file_name)

    float_cols = ['iou_mean','iou_std','recall_mean','recall_std','precision_mean','precision_std','precision_omega_mean','precision_omega_std']

    for col in float_cols: df[col] = pd.to_numeric(df[col],errors='coerce')

    df1 = df.sort_values(by='iou_mean',ascending=False).reset_index(drop=True)

    df2 = df.sort_values(by='recall_mean',ascending=False).reset_index(drop=True)

    df3 = df.sort_values(by='precision_mean',ascending=False).reset_index(drop=True)

    df4 = df.sort_values(by='precision_omega_mean',ascending=False).reset_index(drop=True)

    print_top(df1, 'IoU Mean')
    print_top(df2,'Recall Mean')
    print_top(df3,'Precision Mean')
    print_top(df4,'Precision Ωmega Mean')

    headers = ["chunker", "embedding_function", "config", "iou_mean", "iou_std", "recall_mean", "recall_std", "precision_mean", "precision_std",'precision_omega_mean','precision_omega_std']
    table = []

    for i in range(20):
        row = [
            df1['chunker'][i],
            df1['embedding_function'][i],
            df1['config'][i],
            round(df1['iou_mean'][i] * 100, 3),
            round(df1['iou_std'][i] * 100, 3),
            round(df1['recall_mean'][i] * 100, 3),
            round(df1['recall_std'][i] * 100, 3),
            round(df1['precision_mean'][i] * 100, 3),
            round(df1['precision_std'][i] * 100, 3),
            round(df1['precision_omega_mean'][i] * 100, 3),
            round(df1['precision_omega_std'][i] * 100, 3)
        ]
        table.append(row)

    print(tabulate(table, headers=headers, tablefmt="grid"))


In [11]:
showcase_result(path_folder+'chunker_and_embedding_selection//Chunking_Embedding_eval_trained_embeddings_Greek Cybercrime Legislation.csv')

c://Users//johnk//Documents//GitHub//AILA-application//backend//evaluation//chunker_and_embedding_selection//Chunking_Embedding_eval_trained_embeddings_Greek Cybercrime Legislation.csv
------------------------------------------------------------------------------------------------------------
🏆 Top by IoU Mean:
RecursiveCharacterChunker | IoannisKat1/bge-m3-ft-new | RecursiveCharacterChunker_400_200_IoannisKat1/bge-m3-ft-new
IOU Mean: 7.716%
IOU Std: 10.225%
Recall Mean: 34.697%
Recall Std: 42.087%
Precision Mean: 8.236%
Precision Std: 10.715%
Precision Ω Mean: 57.71%
Precision Ω Std: 26.584%
------------------------------------------------------------------------------------------------------------------------
🏆 Top by Recall Mean:
SentenceChunker | IoannisKat1/all-MiniLM-L6-v2-ft-new | SentenceChunker_20_8191_IoannisKat1/all-MiniLM-L6-v2-ft-new
IOU Mean: 2.816%
IOU Std: 2.073%
Recall Mean: 97.857%
Recall Std: 14.481%
Precision Mean: 2.816%
Precision Std: 2.073%
Precision Ω Mean: 17.0